In [29]:
library(data.table)
library(dplyr)

In [16]:
caqtl_dir = '/directflow/SCCGGroupShare/projects/angxue/proj/tenk10k/caQTL/ms_tables/'
caqtl_summary_file = paste0(caqtl_dir,'TenK10K.caQTL.1Mb.final_summary_qval005.csv')
caqtl_df = as.data.frame(fread(caqtl_summary_file))
head(caqtl_df,2)

,phenotype_id,num_var,beta_shape1,beta_shape2,true_df,pval_true_df,variant_id,start_distance,end_distance,ma_samples,ma_count,af,pval_nominal,slope,slope_se,pval_perm,pval_beta,qval,pval_nominal_threshold,cell_types
,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,chr1:8647508-8648197,6236,0.8629746,156.70200,248.5109,7.910841e-10,1:8157894:GAAGAAGGAGAAGAAGAAGGAAGAGGAAGAGGAGGAGGAAGAAGAAAGAAGGAGGAGGAGGAGAAGGAGA:G,-489959,-489959,5,5,0.05813954,5.08339e-17,4.172526,0.478550,0.00039996,1.153398e-06,0.0014592355,1.152516e-07,ASDC
2,chr1:31903399-31906277,4245,0.7340282,68.03604,225.0327,2.700125e-11,1:32708533:AT:A,803694,803694,1,1,0.07142857,1.28413e-21,10.108527,1.005987,0.00389961,4.220134e-07,0.0007773487,3.712985e-08,ASDC


In [17]:
celltypes = unique(caqtl_df$cell_types)
length(celltypes)
celltypes

[1] 26

[1] "ASDC"              "B_intermediate"    "B_memory"         
 [4] "B_naive"           "CD14_Mono"         "CD16_Mono"        
 [7] "CD4_CTL"           "CD4_Naive"         "CD4_Proliferating"
[10] "CD4_TCM"           "CD4_TEM"           "CD8_Naive"        
[13] "CD8_TCM"           "CD8_TEM"           "HSPC"             
[16] "MAIT"              "NK"                "NK_CD56bright"    
[19] "NK_Proliferating"  "Plasmablast"       "Treg"             
[22] "cDC1"              "cDC2"              "dnT"              
[25] "gdT"               "pDC"

In [42]:
atac_pseudobulk_dir = '/directflow/SCCGGroupShare/projects/angxue/proj/tenk10k/caQTL/data/ca_pseudobulk_matrix/'

In [43]:
df_list = c()
for (celltype in celltypes){
    ct_file = paste0(atac_pseudobulk_dir, celltype,'_PseudobulkMatStandardised.csv')
    ct_df = as.data.frame(fread(ct_file))
    rownames(ct_df) = ct_df$loc
    ct_df$loc = c()
    df_curr = data.frame(peak=rownames(ct_df))
    df_curr[[paste0('is_open_',celltype)]] = 1
    df_curr[[paste0('mean_peak_',celltype)]] = rowMeans(ct_df)
    df_list[[celltype]] = df_curr
}

In [45]:
df_all = data.frame(peak='chr1:8647508-8648197')
for (celltype in celltypes){
    df_all = merge(df_all,df_list[[celltype]], by='peak', all = TRUE)
}

In [46]:
nrow(df_all)
head(df_all)

[1] 440962

,peak,is_open_ASDC,mean_peak_ASDC,is_open_B_intermediate,mean_peak_B_intermediate,is_open_B_memory,mean_peak_B_memory,is_open_B_naive,mean_peak_B_naive,is_open_CD14_Mono,⋯,is_open_cDC1,mean_peak_cDC1,is_open_cDC2,mean_peak_cDC2,is_open_dnT,mean_peak_dnT,is_open_gdT,mean_peak_gdT,is_open_pDC,mean_peak_pDC
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1:100006110-100006783,NA,NA,NA,NA,NA,NA,1,3.244523e-16,1,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,chr1:100010491-100010934,NA,NA,1,-2.351599e-16,1,-8.694070e-18,1,1.727259e-16,1,⋯,NA,NA,1,1.577732e-16,1,2.916236e-17,1,-2.961759e-16,1,-3.374898e-16
3,chr1:100015389-100016239,NA,NA,1,1.707905e-16,1,-3.055252e-16,1,-6.416816e-17,1,⋯,1,1.861519e-17,1,-1.442696e-16,1,2.015857e-16,1,6.799215e-17,1,-7.252355e-17
4,chr1:100027173-100027599,NA,NA,1,-1.597170e-16,1,-4.481034e-16,1,2.404364e-16,1,⋯,NA,NA,1,5.682372e-17,1,1.327855e-16,1,3.094548e-16,NA,NA
5,chr1:100028265-100029490,NA,NA,1,7.817016e-17,1,1.116563e-16,1,-7.132666e-18,1,⋯,NA,NA,1,6.411840e-17,1,-4.917596e-17,1,1.146851e-16,1,1.351051e-16
6,chr1:100034534-100034790,NA,NA,1,-6.695847e-18,1,-7.319864e-17,1,-1.508564e-17,1,⋯,NA,NA,1,-2.308620e-16,1,-1.820249e-17,1,6.585191e-17,1,1.793689e-16


In [47]:
df_all[is.na(df_all)] <- 0
head(df_all)

,peak,is_open_ASDC,mean_peak_ASDC,is_open_B_intermediate,mean_peak_B_intermediate,is_open_B_memory,mean_peak_B_memory,is_open_B_naive,mean_peak_B_naive,is_open_CD14_Mono,⋯,is_open_cDC1,mean_peak_cDC1,is_open_cDC2,mean_peak_cDC2,is_open_dnT,mean_peak_dnT,is_open_gdT,mean_peak_gdT,is_open_pDC,mean_peak_pDC
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1:100006110-100006783,0,0,0,0.000000e+00,0,0.000000e+00,1,3.244523e-16,1,⋯,0,0.000000e+00,0,0.000000e+00,0,0.000000e+00,0,0.000000e+00,0,0.000000e+00
2,chr1:100010491-100010934,0,0,1,-2.351599e-16,1,-8.694070e-18,1,1.727259e-16,1,⋯,0,0.000000e+00,1,1.577732e-16,1,2.916236e-17,1,-2.961759e-16,1,-3.374898e-16
3,chr1:100015389-100016239,0,0,1,1.707905e-16,1,-3.055252e-16,1,-6.416816e-17,1,⋯,1,1.861519e-17,1,-1.442696e-16,1,2.015857e-16,1,6.799215e-17,1,-7.252355e-17
4,chr1:100027173-100027599,0,0,1,-1.597170e-16,1,-4.481034e-16,1,2.404364e-16,1,⋯,0,0.000000e+00,1,5.682372e-17,1,1.327855e-16,1,3.094548e-16,0,0.000000e+00
5,chr1:100028265-100029490,0,0,1,7.817016e-17,1,1.116563e-16,1,-7.132666e-18,1,⋯,0,0.000000e+00,1,6.411840e-17,1,-4.917596e-17,1,1.146851e-16,1,1.351051e-16
6,chr1:100034534-100034790,0,0,1,-6.695847e-18,1,-7.319864e-17,1,-1.508564e-17,1,⋯,0,0.000000e+00,1,-2.308620e-16,1,-1.820249e-17,1,6.585191e-17,1,1.793689e-16


In [55]:
write.csv(df_all, '/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/review_files/rare_variants/atac_pseudobulk_overview.csv')